# AarogyaCare-RAG: Knowledge Base Embedding Pipeline

## Overview

This notebook prepares the medical knowledge base for the AarogyaCare RAG chatbot.

The pipeline performs the following steps:

- Load the medical conditions dataset
- Parse individual disease documents
- Generate semantic embeddings using Sentence Transformers
- Upload vectors to Pinecone
- Verify successful indexing
- Test semantic similarity search

The generated vector database is later used by the FastAPI backend and Streamlit frontend to retrieve relevant medical information before generating responses with Google Gemini.

## Step 1: Import Required Libraries

Import all required packages for embedding generation, vector database connectivity, environment variable management, and data preprocessing.

In [15]:
import os
import re
import uuid

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from tqdm import tqdm

## Step 2: Load Configuration

Load API keys and project configuration from the `.env` file and initialize the embedding model and Pinecone client.

In [16]:
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX = os.getenv("PINECONE_INDEX")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

pc = Pinecone(api_key=PINECONE_API_KEY)

index = pc.Index(PINECONE_INDEX)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


## Step 3: Load Medical Dataset

Read the disease knowledge base from the local dataset.

Each disease contains structured information such as:

- Condition Name
- Home Remedies
- Symptoms
- Risk Factors
- Prevention

The complete text for each disease will be embedded for semantic retrieval.

In [17]:
with open("C:/Users/affaa/OneDrive/Desktop/AarogyaCare-RAG/data/disease_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))

1337610


## Step 4: Parse Disease Documents

Split the dataset into individual disease records.

Each disease is assigned:

- A unique identifier (UUID)
- Disease name
- Complete medical document

This structure will later be stored in Pinecone.

In [18]:
sections = re.split(r"\nCondition\s*\n", text)

sections = [s.strip() for s in sections if s.strip()]

print("Diseases:", len(sections))

Diseases: 371


In [20]:
import uuid

diseases = []

for section in sections:

    lines = [line.strip() for line in section.split("\n") if line.strip()]

    # Skip the word "Condition" if present
    if lines[0].lower() == "condition":
        disease_name = lines[1]
    else:
        disease_name = lines[0]

    diseases.append({
        "id": str(uuid.uuid4()),
        "disease": disease_name,
        "text": section
    })

print(diseases[0])

{'id': 'f09c45d9-c16f-4144-82c3-1087b17a19b4', 'disease': 'Abdominal aortic aneurysm', 'text': "Condition \nAbdominal aortic aneurysm\nHome Remedies\nYour health care provider may tell you to avoid heavy lifting and vigorous physical activity. These activities may cause extreme increases in blood pressure, which can worsen an aneurysm.\nEmotional stress also can raise blood pressure. Try to avoid conflict and stressful situations. If you're feeling stressed or anxious, let your care provider know. Together you can come up with the best treatment plan.\nSymptoms=Abdominal aortic aneurysms often grow slowly without noticeable symptoms. This makes them difficult to detect. Some aneurysms never rupture. Many start small and stay small. Others grow larger over time, sometimes quickly.\nIf you have a growing abdominal aortic aneurysm, you might notice:\nDeep, constant pain in the belly area or side of the belly.\nBack pain.\nA pulse near the bellybutton.\nRisk Factors\nAbdominal aortic aneur

In [21]:
print(diseases[0]["disease"])
print(diseases[1]["disease"])
print(diseases[2]["disease"])
print(len(diseases))

Abdominal aortic aneurysm
Absence seizure
Acne
371


## Step 5: Generate Semantic Embeddings

Generate dense vector embeddings using the Sentence Transformers model:

**sentence-transformers/all-MiniLM-L6-v2**

The model converts each disease document into a 384-dimensional vector representing its semantic meaning.

These vectors enable similarity search instead of traditional keyword matching.

In [22]:
texts = [d["text"] for d in diseases]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding Shape: (371, 384)


In [23]:
vectors = []

for disease, embedding in zip(diseases, embeddings):

    vectors.append({
        "id": disease["id"],
        "values": embedding.tolist(),
        "metadata": {
            "disease": disease["disease"],
            "text": disease["text"]
        }
    })

print(f"Prepared {len(vectors)} vectors.")

Prepared 371 vectors.


## Step 6: Upload Embeddings to Pinecone

Convert embeddings into Pinecone vector format and upload them in batches.

Each vector contains:

- Unique ID
- Embedding values
- Disease name
- Complete medical document

Batch uploading improves efficiency and scalability.

In [24]:
from tqdm import tqdm

BATCH_SIZE = 50

for i in tqdm(range(0, len(vectors), BATCH_SIZE)):
    batch = vectors[i:i+BATCH_SIZE]
    index.upsert(vectors=batch)

print("✅ Upload Complete!")

100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:06<00:00,  1.20it/s]

✅ Upload Complete!


## Step 7: Verify Vector Database

Verify that all vectors have been successfully uploaded to Pinecone by checking:

- Vector count
- Embedding dimension
- Similarity metric

In [25]:
stats = index.describe_index_stats()

print(stats)

DescribeIndexStatsResponse(dimension=384, total_vector_count=371, metric='cosine', namespaces=1)


## Step 8: Test Semantic Retrieval

Perform a semantic similarity search using a sample medical query.

This validates that:

- Query embeddings are generated correctly
- Pinecone retrieves relevant diseases
- The knowledge base is ready for Retrieval-Augmented Generation (RAG)

In [26]:
query = "I have severe back pain"

query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True
).tolist()

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True
)

for match in results["matches"]:
    print("="*80)
    print("Disease:", match["metadata"]["disease"])
    print("Score:", round(match["score"],4))
    print()

Disease: Back Pain
Score: 0.6767

Disease: Spinal stenosis
Score: 0.6164

Disease: Sacroiliitis
Score: 0.4936



# Conclusion

The medical knowledge base has been successfully indexed into Pinecone.

## Summary

- Dataset parsed successfully
- Semantic embeddings generated locally
- 371 medical conditions indexed
- Pinecone vector database verified
- Semantic retrieval tested successfully

This vector database serves as the retrieval layer for the AarogyaCare RAG chatbot. During runtime, user queries are embedded locally, matched against Pinecone, and the retrieved medical context is provided to Google Gemini to generate accurate and context-aware responses.